In [ ]:
import numpy as np
import nibabel as nib
from tqdm import tqdm
import torch # Added PyTorch imports here for completeness
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from sklearn.model_selection import train_test_split

# --- HELPER FUNCTION: to_channels ---
def to_channels ( arr : np . ndarray , dtype = np . uint8 ) -> np . ndarray :
    channels = np . unique ( arr )
    res = np . zeros ( arr . shape + ( len ( channels ) ,) , dtype = dtype )
    for c in channels :
        c = int( c )
        # NOTE: Using the locally defined function name here instead of 'utils.to_channels'
        res [..., c : c +1][ arr == c ] = 1
    return res

# --- NIFTI LOADING FUNCTION: load_data_2D ---
def load_data_2D ( imageNames , normImage = False , categorical = False , dtype = np . float32 , 
getAffines = False , early_stop = False ) :
    '''
    Load medical image data from names, cases list provided into a list for each.
    '''
    affines = []
    
    # get fixed size
    num = len( imageNames )
    first_case = nib . load ( imageNames [0]) . get_fdata ( caching = 'unchanged ')
    if len( first_case . shape ) == 3:
        first_case = first_case [: ,: ,0] # sometimes extra dims , remove
    if categorical :
        first_case = to_channels ( first_case , dtype = dtype )
        rows , cols , channels = first_case . shape
        images = np . zeros (( num , rows , cols , channels ) , dtype = dtype )
    else :
        rows , cols = first_case . shape
        images = np . zeros (( num , rows , cols ) , dtype = dtype )
    
    for i , inName in enumerate ( tqdm ( imageNames ) ) :
        niftiImage = nib . load ( inName )
        inImage = niftiImage . get_fdata ( caching = 'unchanged ') # read disk only
        affine = niftiImage . affine
        if len( inImage . shape ) == 3:
            inImage = inImage [: ,: ,0] # sometimes extra dims in HipMRI_study data

        inImage = inImage . astype ( dtype )
        if normImage :
            inImage = ( inImage - inImage . mean () ) / inImage . std ()
        if categorical :
            # NOTE: Using the locally defined function name here
            inImage = to_channels ( inImage , dtype = dtype ) 
            images [i ,: ,: ,:] = inImage
        else :
            images [i ,: ,:] = inImage
        
        affines . append ( affine )
        if i > 20 and early_stop :
            break
    
    if getAffines :
        return images , affines
    else :
        return images

In [ ]:
# --- 1. Data Loading and Preprocessing ---

# IMPORTANT: REPLACE these with the actual local paths to your downloaded NIfTI files!
image_files = ['path/to/image_001.nii.gz', 'path/to/image_002.nii.gz', ...] 
label_files = ['path/to/label_001.nii.gz', 'path/to/label_002.nii.gz', ...]

# Load Images (NumPy array: [N, H, W] or [N, H, W, 1])
X_data = load_data_2D(image_files, normImage=True, categorical=False) 

# Load Labels (NumPy array: [N, H, W, C]) 
Y_data = load_data_2D(label_files, normImage=False, categorical=True, dtype=np.uint8)

# Add channel dimension to X_data if missing, and convert to PyTorch Channel-First [N, C, H, W]
if len(X_data.shape) == 3:
    X_data = np.expand_dims(X_data, axis=-1) 

X_data = np.transpose(X_data, (0, 3, 1, 2))
Y_data = np.transpose(Y_data, (0, 3, 1, 2))

# --- 2. PyTorch Dataset and Dataloaders ---

# ... (Paste the MRIDataset class definition, data splitting, and DataLoader setup here) ...


